# Lesson 4f — Predict the next word (the ChatGPT loop)

This is the **runnable companion** to the *"data journey"* widget in the Lesson 4 webpage.

In the main Lesson 4 we trained **TinyMLM**, a *fill-in-the-blank* model (BERT-style): it sees a sentence with a `<mask>` in the middle and guesses the missing word. It can peek at words on **both sides** of the gap.

Here we build the *other* flavour — the one that makes **ChatGPT** feel alive:

> **predict-the-next-word** (GPT-style). The model only sees the words **before** the gap, so it can keep going forever: predict a word, add it, predict again, repeat.

The pipeline is *identical* (embed → attention → feed-forward → head → softmax). The only change is **where the blank is**: in the middle (BERT) or always at the end (GPT). We'll also print the **data journey** — every stage the words pass through — for any sentence you like.

Same 5-line training loop you've used since Lesson 1. Run the cells top to bottom.

In [1]:
import random, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); random.seed(0)

## 1. A tiny corpus

Real models train on a big chunk of the internet. We'll use ~30 short, kid-friendly sentences. They overlap a lot on purpose (lots of "the cat …", "the dog …", "went to the …") so a tiny model can pick up the patterns.

In [2]:
sentences = [
    "the cat sat on the mat", "the cat sat on the couch", "the dog sat on the mat",
    "the dog likes to run", "the cat likes to sleep", "the dog likes to play",
    "the cat likes to play", "the dog likes to sleep",
    "she opened the door", "she opened the box", "he opened the door", "he opened the box",
    "i went to the park", "i went to the store", "we went to the park", "we went to the store",
    "the dog ran to the park", "the cat ran to the door", "the dog ran to the mat",
    "the boy likes to play", "the girl likes to run", "the boy likes to run", "the girl likes to play",
    "she went to the park", "he went to the store", "the cat sat on the couch",
    "the dog plays in the park", "the cat sleeps on the couch", "the boy went to the store",
]
print(len(sentences), "sentences")

29 sentences


## 2. Vocabulary + special tokens

Every distinct word gets an id (a row number). We add three helpers:

- `<pad>` — filler so all sentences can be the same length in a batch.
- `<bos>` — "beginning of sentence". We start every sentence with it, so the model has *something* to look at when predicting the very first word.
- `<eos>` — "end of sentence". The model learns to predict this when the sentence is done — that's how it knows when to **stop** writing.

In [3]:
words = sorted({w for s in sentences for w in s.split()})
vocab = ["<pad>", "<bos>", "<eos>"] + words
tok2id = {w: i for i, w in enumerate(vocab)}
id2tok = {i: w for w, i in tok2id.items()}
V = len(vocab)
PAD, BOS, EOS = tok2id["<pad>"], tok2id["<bos>"], tok2id["<eos>"]
MAXLEN = 16  # room for <bos> + prompt + generated words

def encode(s):
    return [BOS] + [tok2id[w] for w in s.split()] + [EOS]

print("vocab size V =", V)
print("vocab:", vocab)
print("encode('the cat sat') ->", encode("the cat sat"))

vocab size V = 31
vocab: ['<pad>', '<bos>', '<eos>', 'box', 'boy', 'cat', 'couch', 'dog', 'door', 'girl', 'he', 'i', 'in', 'likes', 'mat', 'on', 'opened', 'park', 'play', 'plays', 'ran', 'run', 'sat', 'she', 'sleep', 'sleeps', 'store', 'the', 'to', 'we', 'went']
encode('the cat sat') -> [1, 27, 5, 22, 2]


## 3. The model — a next-word Transformer

Almost identical to TinyMLM, with two additions for next-word prediction:

1. **A position embedding** (`self.pos`). Attention alone doesn't know word *order*; we add a per-slot vector so the model can tell "the cat" from "cat the".
2. **A causal mask.** This is the key trick. It blocks every token from attending to words *to its right* — so when predicting word 5, the model is only allowed to look at words 1–4. That's what makes it a left-to-right writer instead of a fill-in-the-blank reader.

Everything else — embedding, the encoder layer (attention + feed-forward), the linear head — is exactly what you've already built.

In [4]:
class NextWord(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.tok  = nn.Embedding(V, d)        # word id   -> vector        (L2)
        self.pos  = nn.Embedding(MAXLEN, d)   # position  -> vector        (order)
        self.enc  = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=64,
                                               batch_first=True)  # attention + FFN (L3 + L1.5)
        self.head = nn.Linear(d, V)           # vector    -> score per word

    def hidden(self, x):
        T = x.size(1)
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.tok(x) + self.pos(pos)                       # embed + position
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)  # causal: no peeking right
        return self.enc(h, src_mask=mask)                     # contextualised vectors

    def forward(self, x):
        return self.head(self.hidden(x))                      # scores for every position

model = NextWord()
print("parameters:", sum(p.numel() for p in model.parameters()))

parameters: 11071


## 4. Train: predict each word from the ones before it

The trick for making training data is even simpler than masking. For a sentence `<bos> the cat sat on the mat <eos>`:

- **input**  = everything except the last token
- **target** = everything except the first token (shifted left by one)

So at each position the model must predict *the next* token. `<bos> the cat …` → predict `the cat sat …`. We score with the same **cross-entropy** from Step 4 of the main lesson, ignoring `<pad>` positions.

In [5]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

# pad all sentences to the same length -> one batch
rows = [encode(s) for s in sentences]
L = max(len(r) for r in rows)
X = torch.full((len(rows), L), PAD)
for i, r in enumerate(rows):
    X[i, :len(r)] = torch.tensor(r)

for step in range(800):
    inp, tgt = X[:, :-1], X[:, 1:]                  # shift: predict the NEXT token
    logits = model(inp)                              # 1. guess
    loss = F.cross_entropy(logits.reshape(-1, V),
                           tgt.reshape(-1),
                           ignore_index=PAD)         # 2. how wrong?
    opt.zero_grad()                                  # 3. clear
    loss.backward()                                  # 4. gradients
    opt.step()                                       # 5. nudge
    if step % 100 == 0:
        print(f"  step {step:3d}   loss {loss.item():.3f}")

  step   0   loss 3.612
  step 100   loss 0.616


  step 200   loss 0.577
  step 300   loss 0.565
  step 400   loss 0.542


  step 500   loss 0.542
  step 600   loss 0.550
  step 700   loss 0.552


The loss settles around ~0.5 and **won't** hit zero — that's correct and honest. After "the" the next word is genuinely ambiguous (mat? couch? dog? park?), so the model *can't* be 100% sure. A perfect-zero loss would mean it had simply memorised, not learned a pattern.

## 5. Generate — the actual ChatGPT loop

Now the payoff. To write text we:

1. Run the model on the words so far.
2. Take the prediction for the **last** position (the next word).
3. **Append** it.
4. Repeat — feeding the model its own output — until it predicts `<eos>`.

That's it. That's how every chatbot writes: one word at a time, each new word chosen using everything written so far. This is *exactly* what the "➕ add & continue" button does in the webpage widget.

First we switch the model to **evaluation mode** with `model.eval()`. During training the encoder uses *dropout* (randomly ignoring some signals to avoid memorising); for clean, repeatable predictions we turn that off.

In [6]:
model.eval()  # inference mode: turn off dropout so predictions are clean & repeatable

@torch.no_grad()
def generate(prompt, max_new=10, greedy=True):
    ids = [BOS] + [tok2id[w] for w in prompt.split()]
    for _ in range(max_new):
        if len(ids) >= MAXLEN: break
        logits = model(torch.tensor([ids]))[0, -1]    # scores for the next word
        probs = F.softmax(logits, dim=-1)
        nxt = int(probs.argmax()) if greedy else int(torch.multinomial(probs, 1))
        if nxt in (PAD, EOS): break                    # model decided to stop
        ids.append(nxt)
    return " ".join(id2tok[i] for i in ids[1:])        # drop <bos>

for p in ["the cat", "the dog likes to", "she opened the", "i went to the"]:
    print(f"  {p:22s} -> {generate(p)}")

  the cat                -> the cat sat on the couch
  the dog likes to       -> the dog likes to play
  she opened the         -> she opened the door
  i went to the          -> i went to the park


Coherent little sentences, written one word at a time by a model with ~11,000 parameters. PRAGMA-Large has **1,000,000,000** — same loop, vastly more patterns learned.

**Try `greedy=False`** below: instead of always taking the single most-likely word, it *samples* from the probabilities. That's the "temperature/randomness" knob — it's why ChatGPT gives a different answer each time you ask.

In [7]:
for _ in range(4):
    print("  sampled:", generate("the dog", greedy=False))

  sampled: the dog ran to the park
  sampled: the dog likes to play
  sampled: the dog plays in the park
  sampled: the dog ran to the mat


## 6. The data journey — every stage, for one sentence

This prints exactly what the webpage widget animates: the words flowing through **tokenise → embed → attention → head → softmax**, ending in the next-word probabilities. All numbers are **real**, read straight out of the trained model.

In [8]:
@torch.no_grad()
def journey(prompt, k=5):
    ids = [BOS] + [tok2id[w] for w in prompt.split()]
    x = torch.tensor([ids]); T = x.size(1)
    toks = [id2tok[i] for i in ids]

    print(f"DATA JOURNEY for: {prompt!r}\n")
    print("Stage 1 — tokenise")
    print("   words:", toks)
    print("   ids:  ", ids)

    pos = torch.arange(T).unsqueeze(0)
    h = model.tok(x) + model.pos(pos)
    print("\nStage 2 — embed (first 4 of 32 dims)")
    for i, t in enumerate(toks):
        print(f"   {t:6s}", [round(float(v), 2) for v in h[0, i, :4]])

    mask = nn.Transformer.generate_square_subsequent_mask(T)
    _, attn = model.enc.self_attn(h, h, h, attn_mask=mask,
                                  need_weights=True, average_attn_weights=True)
    print("\nStage 3 — attention: how much the LAST word looks at each word")
    for i, t in enumerate(toks):
        w = float(attn[0, -1, i]) * 100
        print(f"   {t:6s} {w:5.1f}%  {'#' * int(w/3)}")

    logits = model(x)[0, -1]
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    print("\nStages 4-5 — head -> scores -> softmax: top next-word predictions")
    for p, idx in zip(top.values, top.indices):
        pct = float(p) * 100
        print(f"   {id2tok[int(idx)]:6s} {pct:5.1f}%  {'=' * int(pct/3)}")

journey("the cat sat on the")

DATA JOURNEY for: 'the cat sat on the'

Stage 1 — tokenise
   words: ['<bos>', 'the', 'cat', 'sat', 'on', 'the']
   ids:   [1, 27, 5, 22, 15, 27]

Stage 2 — embed (first 4 of 32 dims)
   <bos>  [-1.7, 0.48, -2.54, 1.13]
   the    [2.11, -1.14, -1.68, -1.71]
   cat    [0.85, -0.73, -1.49, 0.9]
   sat    [-0.78, 1.64, 0.58, -3.12]
   on     [-1.24, -0.18, -0.07, -1.34]
   the    [3.23, -1.44, -0.98, -0.12]

Stage 3 — attention: how much the LAST word looks at each word
   <bos>    0.3%  
   the      1.3%  
   cat     22.9%  #######
   sat      0.3%  
   on      73.7%  ########################
   the      1.6%  

Stages 4-5 — head -> scores -> softmax: top next-word predictions
   couch   66.9%  ======================
   mat     32.9%  ==========
   store    0.0%  
   we       0.0%  
   the      0.0%  


Read it top to bottom and compare with the widget: the last word's attention concentrates on the **content** words that decide the answer (here `cat` and `on`), and the head turns the resulting summary into a confident next-word guess (`couch` / `mat`).

Try another:

In [9]:
journey("she opened the")

DATA JOURNEY for: 'she opened the'

Stage 1 — tokenise
   words: ['<bos>', 'she', 'opened', 'the']
   ids:   [1, 23, 16, 27]

Stage 2 — embed (first 4 of 32 dims)
   <bos>  [-1.7, 0.48, -2.54, 1.13]
   she    [-0.87, 0.11, 0.45, -0.76]
   opened [-0.99, -0.16, -2.33, 1.6]
   the    [1.85, -0.1, -2.03, -2.44]

Stage 3 — attention: how much the LAST word looks at each word
   <bos>    0.9%  
   she      2.1%  
   opened  73.7%  ########################
   the     23.3%  #######

Stages 4-5 — head -> scores -> softmax: top next-word predictions
   door    51.2%  =================
   box     48.6%  ================
   park     0.0%  
   to       0.0%  
   in       0.0%  


## 7. Now you experiment

1. **Add a new sentence** to `sentences` (e.g. `"the bird likes to fly"`), re-run from cell 1, and see if `generate("the bird")` learns it. (You'll also need `fly`/`bird` to appear — add a couple of sentences using them.)
2. **Shrink the model**: set `d=8` in `NextWord`. Does it still generate clean sentences? How low can you go?
3. **Turn off the causal mask** (pass `src_mask=None` in `hidden`). Generation breaks — why? (Hint: the model can now "cheat" by looking at the word it's supposed to predict during training.)
4. **Crank the randomness**: call `generate(..., greedy=False)` ten times on the same prompt. Count how many *different* sentences you get. That variety is the model's "creativity".
5. Run `journey()` on a prompt the model is *unsure* about (e.g. `"the"`) and watch the probabilities spread out instead of spiking — that's the model honestly saying "could be lots of things".